In [ ]:
# Environment Setup
import os
import sys
import json
import time
import asyncio
from datetime import datetime
from typing import Dict, List, Optional, Any

# Load environment variables
import dotenv

dotenv.load_dotenv()

# For making HTTP requests
import httpx

# API Configuration
API_URL = "https://openrouter.ai/api/v1/chat/completions"
HEADERS = {
    "Authorization": f"Bearer {os.getenv('OPENROUTER_API_KEY')}",
    "Content-Type": "application/json",
}
DEFAULT_MODEL = "anthropic/claude-3.5-haiku"
DEFAULT_TEMPERATURE = 0.7
DEFAULT_MAX_TOKENS = 2048
TIMEOUT = 60.0
# DEFAULT_TEMPERATURE 决定了 AI “怎么说话”（是老实交代还是天马行空）。
# TIMEOUT 决定了你的程序 “等多久”（是耐心等待还是及时止损）。

print("✅ Environment ready!")
print(f"📍 Using model: {DEFAULT_MODEL}")
print(f"🔑 API key loaded: {'Yes' if os.getenv('OPENROUTER_API_KEY') else 'No'}")

In [ ]:
def invokeModel(prompt: str, model: str = DEFAULT_MODEL) -> str:
    """Simple synchronous LLM invocation."""
    response = httpx.post(
        API_URL,
        headers=HEADERS,
        json={"model": model, "messages": [{"role": "user", "content": prompt}]},
        timeout=TIMEOUT,
    )
    return response.json()["choices"][0]["message"]["content"]


def parseSSE(response, handler):
    """Parse Server-Sent Events for streaming responses.
    当模型一点点返回文本时，它能逐步接收、处理每个片段。
    """
    buffer = ""
    for chunk in response.iter_text():
        buffer += chunk
        while "\n" in buffer:
            line, buffer = buffer.split("\n", 1)
            if line.startswith("data: "):  # SSE data line
                data = line[6:]
                if data == "[DONE]":
                    return
                try:
                    handler(json.loads(data))
                except:
                    pass


def invokeModelStream(prompt: str, model: str = DEFAULT_MODEL):
    """Stream tokens as they arrive for better UX.
    不同于上面的 invokeModel()，它不会等整段文字生成完毕，而是实时输出 token
    """
    with httpx.stream(
        "POST",
        API_URL,
        headers=HEADERS,
        json={
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
            "stream": True,
            "temperature": DEFAULT_TEMPERATURE,
            "max_output_tokens": DEFAULT_MAX_TOKENS,
        },
        timeout=TIMEOUT,
    ) as response:
        parseSSE(
            response,
            lambda data: (
                print(
                    data["choices"][0]["delta"].get("content", ""), end="", flush=True
                )
                if "choices" in data
                else None
            ),
        )


print("✅ LLM functions defined!")

In [ ]:
# Define tool functions - just regular Python!
def get_current_time():
    """Tool function that returns current time with seconds."""
    now = datetime.now()
    return f"Current time: {now.strftime('%I:%M:%S %p on %A, %B %d, %Y')}"


def get_current_day():
    """Tool function that returns current day of the week."""
    now = datetime.now()
    return f"Today is {now.strftime('%A, %B %d, %Y')}"


# Test our tools locally
print("Testing tools:")
print(f"  - {get_current_time()}")
print(f"  - {get_current_day()}")

In [ ]:
# Define tool schemas in the open router format which uses the OpenAI function calling spec
# We need this because we don't want to do any "smart stuff" on the model's outputs
# let the model do its own smart stuff and give us exact params

TIME_TOOL = {
    "type": "function",
    "function": {
        "name": "get_current_time",
        "description": "Get the current time with seconds",
        "parameters": {"type": "object", "properties": {}},
    },
}

DAY_TOOL = {
    "type": "function",
    "function": {
        "name": "get_current_day",
        "description": "Get the current day of the week",
        "parameters": {"type": "object", "properties": {}},
    },
}

print("✅ Tool schemas defined!")
print("📋 Available tools: get_current_time, get_current_day")

In [ ]:
# Implement tool calling with conversation memory
# User → Model → Tool → Model → Output

conversation_history = []


def invokeModelWithTools(prompt, tools=None, model=DEFAULT_MODEL):
    """Tool calling with streaming final response and conversation memory."""
    global conversation_history
    tools_map = {
        "get_current_time": get_current_time,
        "get_current_day": get_current_day,
    }

    # Add new user message to history
    conversation_history.append({"role": "user", "content": prompt})

    # Step 1: Get tool calls
    response = httpx.post(
        API_URL,
        headers=HEADERS,
        json={"model": model, "messages": conversation_history, "tools": tools},
        timeout=TIMEOUT,
    ).json()

    assistant_msg = response["choices"][0]["message"]
    conversation_history.append(assistant_msg)

    # Step 2: Execute tool calls
    if assistant_msg.get("tool_calls"):
        for tool_call in assistant_msg["tool_calls"]:
            tool_name = tool_call["function"]["name"]
            print(f"> 🔧 Calling tool: {tool_name}")
            if tool_name in tools_map:
                result = tools_map[tool_name]()
                conversation_history.append(
                    {"role": "tool", "tool_call_id": tool_call["id"], "content": result}
                )

    # Step 3: Stream final response
    final_content = ""

    def capture_content(data):
        nonlocal final_content
        content = data["choices"][0]["delta"].get("content", "")
        final_content += content
        print(content, end="", flush=True)

    with httpx.stream(
        "POST",
        API_URL,
        headers=HEADERS,
        json={
            "model": model,
            "messages": conversation_history,
            "tools": tools,
            "stream": True,
        },
        timeout=TIMEOUT,
    ) as response:
        parseSSE(response, capture_content)

    # Add final response to history
    conversation_history.append({"role": "assistant", "content": final_content})
    print()


print("✅ Tool calling implemented!")

In [ ]:
# A simple weather function - our core business logic
def fetch_weather_data(city: str) -> str:
    """The actual weather fetching logic - identical everywhere."""
    # Using wttr.in for simple weather data
    try:
        response = httpx.get(f"http://wttr.in/{city}?format=j1", timeout=5.0)
        data = response.json()
        current = data["current_condition"][0]
        return f"Weather in {city}: {current['weatherDesc'][0]['value']}, {current['temp_F']}°F"
    except:
        return f"Could not get weather for {city}"


# Test it
print("🌤️  Testing our weather function:")
print(fetch_weather_data("New York"))

In [ ]:
# First, let's import FastMCP
from fastmcp import FastMCP, Context
from pydantic import BaseModel, Field
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)
import nest_asyncio

nest_asyncio.apply()

from src.server.services.http_client import BaseHTTPClient

HN_API_BASE = "https://hacker-news.firebaseio.com/v0"
HN_WEBSITE_BASE = "https://news.ycombinator.com"


class HackerNewsClient(BaseHTTPClient):
    async def get_story_ids(self, endpoint: str, count: int) -> list[int]:
        async def _fetch():
            async with httpx.AsyncClient(**self.client_config) as client:
                response = await client.get(f"{HN_API_BASE}/{endpoint}.json")
                response.raise_for_status()
                return response.json()[:count]  # Limit to requested count

        try:
            ids = await _fetch()
            print(f"Fetched {len(ids)} story IDs from {endpoint}")
            return ids
        except Exception as e:
            print(f"Failed to fetch story IDs from {endpoint}: {e}")
            raise

    async def get_item(self, item_id: int) -> Optional[dict]:
        async def _fetch():
            async with httpx.AsyncClient(**self.client_config) as client:
                response = await client.get(f"{HN_API_BASE}/item/{item_id}.json")
                response.raise_for_status()
                return response.json()

        try:
            item = await _fetch()
            if item:
                print(f"Fetched item {item_id}: {item.get('type', 'unknown')}")
            return item
        except Exception as e:
            print(f"Failed to fetch item {item_id}: {e}")
            return None


# Create our MCP server instance
mcp_v1 = FastMCP(
    name="newspaper-agent-server-v1",
)


@mcp_v1.tool()
# get hacker news stories
async def ghn(count) -> str:
    if not 1 <= count <= 150:
        raise ValueError("Count must be between 1 and 150")

    client = HackerNewsClient()

    # Fetch story IDs from top stories endpoint
    story_ids = await client.get_story_ids("topstories", count)

    # Fetch detailed story information
    stories = []
    for story_id in story_ids:
        story = await client.get_item(story_id)
        if story and story.get("title"):
            stories.append(
                {
                    "title": story["title"],
                    "score": story.get("score", 0),
                    "url": story.get("url", f"{HN_WEBSITE_BASE}/item?id={story_id}"),
                }
            )

    # Format as markdown with numbered list
    result = f"# Top {len(stories)} Hacker News Stories\n\n"
    for i, story in enumerate(stories, 1):
        result += f"{i}. **{story['title']}** ({story['score']} points)\n"
        result += f"   {story['url']}\n\n"

    return result

In [ ]:
from html_to_markdown import convert_to_markdown
from typing import Dict

from src.server.config.constants import DEFAULT_HEADERS


async def fetch(url):
    retries = 2
    async with httpx.AsyncClient(headers=DEFAULT_HEADERS, timeout=120) as client:
        for attempt in range(1, retries + 1):
            try:
                resp = await client.get(url)
                resp.raise_for_status()
                return convert_to_markdown(resp.text)[:5000]
            except httpx.HTTPError as exc:
                if attempt == retries:
                    raise
                await asyncio.sleep(0.5 * attempt)


@mcp_v2.tool()
async def fetch_content(url: str) -> str:
    """
    Fetch HTML from `url` and convert to Markdown.

    Args:
        url: The target URL.
        headers: HTTP headers to spoof a real browser.
        timeout: Request timeout in seconds.
        retries: Number of retry attempts on failure.

    Returns:
        Markdown-converted text.

    Raises:
        httpx.HTTPError: On request failure after retries.
    """
    return await fetch(url)

In [ ]:
# Cell: Phase 3 Imports and Setup
from pathlib import Path
from typing import Dict, List, Optional

# Add src to path for imports
sys.path.insert(0, str(Path.cwd().parent / "src" / "server"))

# Core MCP imports

warnings.filterwarnings("ignore", category=DeprecationWarning)

nest_asyncio.apply()

# Service imports
from services.email_service import EmailService
from services.interests_file import InterestsFileService

# Config imports
from config.settings import get_settings

print("✅ All Phase 3 imports successful!")
print(f"📁 Working directory: {Path.cwd()}")
print(f"📁 Server path: {Path.cwd().parent / 'src' / 'server'}")

In [ ]:
# Cell: Application Context Setup
from contextlib import asynccontextmanager
from dataclasses import dataclass


@dataclass
class AppContext:
    """Application context with all services for newspaper creation."""

    hn_client: HackerNewsClient
    interests_service: InterestsFileService
    email_service: EmailService
    settings: object


@asynccontextmanager
async def app_lifespan(mcp: FastMCP):
    """Initialize all services for the newspaper agent."""
    print("🚀 Starting Newspaper Creation Agent MCP Server")

    # Get settings
    settings = get_settings()
    settings.data_dir.mkdir(parents=True, exist_ok=True)

    # Initialize all services
    hn_client = HackerNewsClient()
    interests_service = InterestsFileService(settings.data_dir)

    # Email service configuration
    email_service = EmailService(
        {
            "server": "smtp.gmail.com",
            "port": 587,
            "use_tls": True,
            "username": "adityaarunsinghal@gmail.com",
            "password": os.getenv("MCP_SMTP_PASSWORD", ""),
            "from_email": "adityaarunsinghal@gmail.com",
            "from_name": "Newspaper Creation Agent",
        }
    )

    print("✅ All services initialized!")
    print(f"📧 Email: {email_service.server}:{email_service.port}")
    print(f"📁 Data: {settings.data_dir}")

    try:
        yield AppContext(
            hn_client=hn_client,
            interests_service=interests_service,
            email_service=email_service,
            settings=settings,
        )
    finally:
        print("👋 Shutting down MCP Server")


print("✅ Application context manager defined!")

In [ ]:
# Cell: MCP Server with Full Tool Suite

mcp = FastMCP(
    name="newspaper-creation-agent",
    instructions="""You are a sophisticated newspaper creation assistant.
    
    Your capabilities:
    - Fetch stories from Hacker News
    - Extract full article content from URLs
    - Track and manage user interests
    - Create structured newspapers with sections
    - Add articles to newspaper sections
    - Send finished newspapers via email
    
    Current date: {date}
    """.format(
        date=datetime.now().strftime("%A, %B %d, %Y")
    ),
    lifespan=app_lifespan,
)

# ============= CORE TOOLS =============


@mcp.tool()
async def fetch_hn_stories(count: int = 5, ctx: Context = None) -> str:
    """Fetch top stories from Hacker News."""
    if not 1 <= count <= 30:
        raise ValueError("Count must be between 1 and 30")

    client = ctx.request_context.lifespan_context.hn_client
    story_ids = await client.get_story_ids("topstories", count)

    stories = []
    for story_id in story_ids:
        story = await client.get_item(story_id)
        if story and story.get("title"):
            stories.append(
                {
                    "id": story_id,
                    "title": story["title"],
                    "score": story.get("score", 0),
                    "url": story.get("url", f"{HN_WEBSITE_BASE}/item?id={story_id}"),
                    "by": story.get("by", "unknown"),
                }
            )

    result = f"# Top {len(stories)} Hacker News Stories\n\n"
    for i, story in enumerate(stories, 1):
        result += (
            f"{i}. **{story['title']}** ({story['score']} points by {story['by']})\n"
        )
        result += f"   {story['url']}\n\n"

    return result


@mcp.tool()
async def fetch_article_content(url: str, ctx: Context = None) -> str:
    """Fetch and extract clean content from a web URL."""
    try:
        content = (await fetch(url))[:8000]
        return f"# Content from {url}\n\n{content}"
    except Exception as e:
        return f"❌ Failed to fetch {url}: {str(e)}"


# ============= NEWSPAPER CREATION TOOLS =============


@mcp.tool()
async def create_newspaper_draft(
    title: str = "Daily Tech Digest",
    subtitle: str = "",
    sections: List[str] = None,
    ctx: Context = None,
) -> str:
    """Create a new newspaper draft with sections."""
    if sections is None:
        sections = ["Top Stories", "Technology", "Analysis"]

    newspaper_id = f"newspaper_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

    draft = {
        "id": newspaper_id,
        "title": title,
        "subtitle": subtitle,
        "created_at": datetime.now().isoformat(),
        "status": "draft",
        "sections": [{"title": sec, "articles": []} for sec in sections],
    }

    # Save draft
    settings = ctx.request_context.lifespan_context.settings
    drafts_dir = Path(settings.data_dir) / "newspapers"
    drafts_dir.mkdir(exist_ok=True)

    draft_file = drafts_dir / f"{newspaper_id}.json"
    with open(draft_file, "w") as f:
        json.dump(draft, f, indent=2)

    # Save HTML draft
    email_service = ctx.request_context.lifespan_context.email_service
    html_content = email_service._create_html_version(draft)
    html_file = drafts_dir / f"{newspaper_id}.html"
    with open(html_file, "w", encoding="utf-8") as f:
        f.write(html_content)

    result = f"# 📰 Newspaper Draft Created\n\n"
    result += f"**ID:** {newspaper_id}\n"
    result += f"**Title:** {title}\n"
    result += f"**Sections:** {', '.join(sections)}\n"
    result += f"**Files:** {draft_file}, {html_file}\n\n"
    result += f"*Use add_article_to_newspaper() to add content*"

    return result


@mcp.tool()
async def add_article_to_newspaper(
    newspaper_id: str,
    section: str,
    title: str,
    content: str,
    url: str = "",
    author: str = "",
    score: int = 10,
    ctx: Context = None,
) -> str:
    """Add an article to a newspaper section with importance score (1-100)."""
    settings = ctx.request_context.lifespan_context.settings
    draft_file = Path(settings.data_dir) / "newspapers" / f"{newspaper_id}.json"

    if not draft_file.exists():
        return f"❌ Newspaper '{newspaper_id}' not found"

    # Load draft
    with open(draft_file, "r") as f:
        draft = json.load(f)

    # Find section
    target_section = None
    for sec in draft["sections"]:
        if sec["title"].lower() == section.lower():
            target_section = sec
            break

    if not target_section:
        sections = [s["title"] for s in draft["sections"]]
        return f"❌ Section '{section}' not found. Available: {', '.join(sections)}"

    # Add article
    article = {
        "title": title,
        "content": content,
        "url": url,
        "author": author,
        "score": score,
        "added_at": datetime.now().isoformat(),
    }
    target_section["articles"].append(article)

    # Save JSON
    with open(draft_file, "w") as f:
        json.dump(draft, f, indent=2)

    # Update HTML
    email_service = ctx.request_context.lifespan_context.email_service
    html_content = email_service._create_html_version(draft)
    html_file = draft_file.with_suffix(".html")
    with open(html_file, "w", encoding="utf-8") as f:
        f.write(html_content)

    total_articles = sum(len(s["articles"]) for s in draft["sections"])

    return f"✅ Added '{title}' to {section}\nTotal articles: {total_articles}"


@mcp.tool()
async def send_newspaper_email(newspaper_id: str, ctx: Context = None) -> str:
    """Send the finished newspaper via email."""
    settings = ctx.request_context.lifespan_context.settings
    draft_file = Path(settings.data_dir) / "newspapers" / f"{newspaper_id}.json"

    if not draft_file.exists():
        return f"❌ Newspaper '{newspaper_id}' not found"

    # Load draft
    with open(draft_file, "r") as f:
        draft = json.load(f)

    # Get email service
    email_service = ctx.request_context.lifespan_context.email_service

    # Send email
    result = email_service.send_newspaper(
        newspaper_data=draft, subject=f"📰 {draft['title']}"
    )

    if result.get("success"):
        return f"✅ Newspaper '{draft['title']}' sent successfully!\n\n{result.get('message', '')}"
    else:
        return f"❌ Failed to send: {result.get('error', 'Unknown error')}"


@mcp.tool()
async def preview_newspaper(newspaper_id: str, ctx: Context = None) -> str:
    """Preview newspaper before sending."""
    settings = ctx.request_context.lifespan_context.settings
    draft_file = Path(settings.data_dir) / "newspapers" / f"{newspaper_id}.json"

    if not draft_file.exists():
        return f"❌ Newspaper '{newspaper_id}' not found"

    with open(draft_file, "r") as f:
        draft = json.load(f)

    result = f"# 📰 {draft['title']}\n"
    if draft.get("subtitle"):
        result += f"*{draft['subtitle']}*\n"
    result += f"\n{datetime.now().strftime('%A, %B %d, %Y')}\n\n"

    for section in draft["sections"]:
        if not section["articles"]:
            continue

        result += f"## {section['title']}\n\n"
        for i, article in enumerate(section["articles"], 1):
            result += f"### {i}. {article['title']}\n"
            if article.get("author"):
                result += f"*By {article['author']}*\n\n"

            # Preview first 200 chars
            preview = article["content"][:200]
            if len(article["content"]) > 200:
                preview += "..."
            result += f"{preview}\n\n"

            if article.get("url"):
                result += f"[Read more]({article['url']})\n\n"
            result += "---\n\n"

    return result


print("✅ All newspaper creation tools registered!")

In [ ]:
# Cell: Resources and Prompts
@mcp.resource("file://interests.md")
async def get_interests_file(ctx: Context = None) -> str:
    """User's interests file."""
    interests_service = ctx.request_context.lifespan_context.interests_service
    interests = interests_service.read_interests()

    result = "# Your Interests\n\n"
    result += f"**Last Updated:** {interests.get('last_updated', 'Unknown')}\n\n"

    topics = interests.get("topics", [])
    if topics:
        result += "## Topics\n"
        for topic in topics:
            result += f"- {topic}\n"

    return result


@mcp.prompt()
async def create_quick_newspaper() -> str:
    """Workflow for creating a quick 5-article newspaper."""
    return """Create a quick newspaper with these steps:

1. fetch_hn_stories(count=10) - Get recent stories
2. create_newspaper_draft(title="Tech Brief", sections=["Headlines", "Deep Dive"])
3. For 2-3 interesting stories:
   - fetch_article_content(url) to get full content
   - add_article_to_newspaper() with summary
4. preview_newspaper() to review
5. send_newspaper_email() to deliver

Keep it concise and focused!"""


@mcp.prompt()
async def create_comprehensive_newspaper() -> str:
    """Workflow for creating a detailed newspaper."""
    return """Create a comprehensive newspaper:

1. fetch_hn_stories(count=20) - Get more options
2. create_newspaper_draft(
    title="Daily Tech Digest",
    sections=["Top Stories", "Technology", "Analysis", "Opinion"]
   )
3. For 8-10 stories across different sections:
   - fetch_article_content(url)
   - Summarize intelligently
   - add_article_to_newspaper() to appropriate section
4. preview_newspaper() for final review
5. send_newspaper_email() to deliver

Aim for depth and variety!"""


print("✅ Resources and prompts registered!")

In [ ]:
# Add prompts - these are like chef's specials!
@mcp.prompt()
async def create_morning_newspaper() -> str:
    """Workflow for creating a morning newspaper.

    Like a prix fixe menu - a curated experience!
    """
    return """Create a comprehensive morning newspaper by:

1. Fetch top stories from Hacker News (fetch_hn_stories)
2. Search for content matching my interests (search_by_interests)
3. Fetch full content for the most relevant articles (fetch_content)
4. Create a newspaper with all content (create_newspaper)
5. Organize into logical sections
6. Ensure each section has 2-3 articles

Make it perfect for a 30-minute commute read!
"""


@mcp.prompt()
async def deep_interest_research() -> str:
    """Deep research based on interests.

    ⚠️ This prompt will cause context overflow! (Intentional for teaching)
    """
    return """For EACH interest in my interests.md file:

1. Search extensively for related content
2. Fetch at least 5 articles per interest
3. Read the full content of each article
4. Find related articles from those articles
5. Create comprehensive summaries
6. Build a mega-newspaper with everything

I want DEEP coverage of each topic. Don't stop until you have 
at least 20 articles total, with full content for each!

This should be incredibly thorough!
"""

In [ ]:
os.system("lsof -ti:8080 | xargs kill -9 2>/dev/null")
await mcp.run_async(transport="streamable-http", port=8080)